Tối ưu Hyperparameter cho XGBoost và Random Forest bằng Cross-Validation trên Spark.

- Train chỉ trên non-zero demand (best practice)
- Dùng Spark ML CrossValidator với ParamGridBuilder
- Metric chính: WAPE (Weighted Absolute Percentage Error)
- Đọc features từ HDFS (output của feature_engineering.ipynb)

In [1]:
%pip install scikit-learn pyarrow statsmodels xgboost


Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import os
import urllib.request
import pandas as pd
import numpy as np

from pyspark.sql import SparkSession, functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import RegressionEvaluator
from xgboost.spark import SparkXGBRegressor

spark = (
    SparkSession.builder
    .appName("HyperparameterTuning_VirtualCluster")
    .master("spark://master:7077")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "hdfs://master:9000/spark-logs")
    .config("spark.driver.memory", "3g")
    .config("spark.driver.memoryOverhead", "1g")
    .config("spark.executor.memory", "5g")
    .config("spark.executor.memoryOverhead", "1g")
    .config("spark.executor.cores", "3")
    .config("spark.cores.max", "9")
    .config("spark.sql.shuffle.partitions", "48")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print("Spark ready:", spark.version)


:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
graphframes#graphframes added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-165fdc0a-8151-4a08-a7e4-3ce32f14a74d;1.0
	confs: [default]
	found graphframes#graphframes;0.8.3-spark3.5-s_2.12 in spark-packages
	found org.slf4j#slf4j-api;1.7.16 in central
:: resolution report :: resolve 135ms :: artifacts dl 2ms
	:: modules in use:
	graphframes#graphframes;0.8.3-spark3.5-s_2.12 from spark-packages in [default]
	org.slf4j#slf4j-api;1.7.16 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   2   |   0   |   0   |   0   ||   2   |   0   |
	-----------------------------------------------

Spark ready: 3.5.0


In [3]:
FEATURE_PATH = "/user/data/feature_engineering/demand_prediction_features"
OUT_BASE_ROOT = "/user/data/results/demand_prediction/tuning"
TARGET_COL = "pickup_demand"
MAPE_THRESHOLD = 5  # Only compute MAPE on demand >= this

feature_cols = [
    "hour", "dow", "month", "is_weekend",
    "lag_3", "lag_6", "lag_168",
    "roll_mean_6", "roll_mean_24", "roll_std_24", "cluster_id",
]

df_all = spark.read.parquet(FEATURE_PATH)
if "split_local" not in df_all.columns and "split" in df_all.columns:
    df_all = df_all.withColumn("split_local", F.col("split"))

train_df = df_all.filter(F.col("split_local") == "train").filter(F.col(TARGET_COL) > 0).cache()
test_df  = df_all.filter(F.col("split_local") == "test").cache()

print(f"Train (non-zero): {train_df.count():,}")
print(f"Test (all):       {test_df.count():,}")


Train (non-zero): 249,461


Test (all):       3,602,628


In [4]:
# ---- Add Holt-Winters forecast feature ----
from pyspark.sql.types import StructType, StructField, DoubleType

HW_SEASONAL_PERIODS = 24
HW_MAX_TRAIN = HW_SEASONAL_PERIODS * 52
HW_MIN_ROWS  = HW_SEASONAL_PERIODS * 3
HW_MIN_TRAIN = HW_SEASONAL_PERIODS * 2

def add_hw_forecast(pdf):
    pdf = pdf.sort_values("pickup_bin_60m").reset_index(drop=True)
    y = pdf["pickup_demand"].astype(float).to_numpy()
    if len(y) < HW_MIN_ROWS:
        pdf["hw_forecast"] = y
        return pdf
    train_mask = pdf["split_local"] == "train"
    y_train_full = y[train_mask.to_numpy()]
    if len(y_train_full) < HW_MIN_TRAIN:
        pdf["hw_forecast"] = y
        return pdf
    try:
        from statsmodels.tsa.holtwinters import ExponentialSmoothing
        y_fit = y_train_full[-HW_MAX_TRAIN:]
        n_fit = len(y_fit)
        n_older = len(y_train_full) - n_fit
        n_test = int((~train_mask).sum())
        hw = ExponentialSmoothing(y_fit, trend="add", seasonal="add",
                                  seasonal_periods=HW_SEASONAL_PERIODS,
                                  initialization_method="estimated")
        fitted = hw.fit(optimized=True)
        in_sample = np.asarray(fitted.fittedvalues)
        train_fc = np.concatenate([np.full(n_older, in_sample[0]), in_sample]) if n_older > 0 else in_sample
        test_fc  = np.asarray(fitted.forecast(n_test)) if n_test > 0 else np.array([])
        forecast = np.concatenate([train_fc, test_fc]) if n_test > 0 else train_fc
        pdf["hw_forecast"] = np.maximum(forecast, 0.0)
    except Exception:
        pdf["hw_forecast"] = y
    return pdf

# Apply on full df (both train + test needed for HW)
df_all_cached = df_all.cache()
hw_schema = StructType(df_all_cached.schema.fields + [StructField("hw_forecast", DoubleType(), True)])
df_hw = df_all_cached.groupBy("PULocationID").applyInPandas(add_hw_forecast, schema=hw_schema).cache()

feature_cols_hw = feature_cols + ["hw_forecast"]

train_hw = df_hw.filter(F.col("split_local") == "train").filter(F.col(TARGET_COL) > 0).cache()
test_hw  = df_hw.filter(F.col("split_local") == "test").cache()
print(f"Train with HW (non-zero): {train_hw.count():,}")
print(f"Test with HW  (all):      {test_hw.count():,}")


Train with HW (non-zero): 249,461


Test with HW  (all):      3,602,628


In [5]:
# ---- Helper: compute metrics on non-zero actual demand ----
rmse_eval = RegressionEvaluator(labelCol=TARGET_COL, predictionCol="prediction", metricName="rmse")
r2_eval   = RegressionEvaluator(labelCol=TARGET_COL, predictionCol="prediction", metricName="r2")

def evaluate(model_name, pred_df):
    eval_df  = pred_df.withColumn("prediction", F.when(F.col("prediction") < 0, F.lit(0.0)).otherwise(F.col("prediction")))
    nonzero  = eval_df.filter(F.col(TARGET_COL) > 0)
    rmse_val = float(rmse_eval.evaluate(nonzero))
    r2_val   = float(r2_eval.evaluate(nonzero))
    mae_val  = float(nonzero.agg(F.avg(F.abs(F.col(TARGET_COL) - F.col("prediction")))).first()[0])
    wape_val = float(nonzero.agg(
        (F.sum(F.abs(F.col(TARGET_COL) - F.col("prediction"))) / F.sum(F.col(TARGET_COL)) * 100.0)
    ).first()[0])
    high_df  = nonzero.filter(F.col(TARGET_COL) >= MAPE_THRESHOLD)
    mape_val = float(high_df.agg(
        F.avg(F.abs(F.col(TARGET_COL) - F.col("prediction")) / F.col(TARGET_COL)) * 100.0
    ).first()[0] or 0)
    print(f"\n=== {model_name} ===")
    print(f"  MAE:      {mae_val:.4f}")
    print(f"  RMSE:     {rmse_val:.4f}")
    print(f"  WAPE:     {wape_val:.2f}%")
    print(f"  MAPE(≥5): {mape_val:.2f}%")
    print(f"  R²:       {r2_val:.4f}")
    return {"model": model_name, "MAE": mae_val, "RMSE": rmse_val,
            "WAPE": wape_val, "MAPE(>=5)": mape_val, "R2": r2_val}

results = []
assembler = VectorAssembler(inputCols=feature_cols_hw, outputCol="features", handleInvalid="skip")


## 1. XGBoost Hyperparameter Tuning

Grid search:
- `max_depth`: độ sâu cây (complexity)
- `learning_rate` (eta): tốc độ học
- `n_estimators`: số cây
- `subsample`: tỷ lệ mẫu mỗi cây
- `min_child_weight`: số mẫu tối thiểu trong leaf

In [6]:
# ---- XGBoost Grid Search ----
xgb = SparkXGBRegressor(
    features_col="features",
    label_col=TARGET_COL,
    prediction_col="prediction",
    num_workers=3,
    objective="reg:squarederror",
    device="cpu",
)

xgb_pipeline = Pipeline(stages=[assembler, xgb])

# Lưới tham số, nhỏ do giới hạn RAM
xgb_grid = (
    ParamGridBuilder()
    .addGrid(xgb.max_depth,      [4, 6, 8])
    .addGrid(xgb.learning_rate,  [0.05, 0.1])
    .addGrid(xgb.n_estimators,   [50, 100])
    .addGrid(xgb.subsample,      [0.7, 0.9])
    .build()
)

print(f"XGBoost grid size: {len(xgb_grid)} combinations × 3 folds = {len(xgb_grid)*3} training runs")

cv_xgb = CrossValidator(
    estimator=xgb_pipeline,
    estimatorParamMaps=xgb_grid,
    evaluator=RegressionEvaluator(labelCol=TARGET_COL, predictionCol="prediction", metricName="rmse"),
    numFolds=3,
    parallelism=2,  # 2 models in parallel
    seed=42,
)

print("Fitting XGBoost CrossValidator... (may take 10–30 minutes)")
cv_xgb_model = cv_xgb.fit(train_hw)

# Best params
best_xgb_params = cv_xgb_model.getEstimatorParamMaps()[int(np.argmin(cv_xgb_model.avgMetrics))]
print("\nBest XGBoost params:")
for k, v in best_xgb_params.items():
    print(f"  {k.name}: {v}")

# Evaluate best model
xgb_pred = cv_xgb_model.transform(test_hw)
xgb_result = evaluate("xgboost_tuned", xgb_pred)
results.append(xgb_result)


XGBoost grid size: 24 combinations × 3 folds = 72 training runs
Fitting XGBoost CrossValidator... (may take 10–30 minutes)


2026-05-08 14:38:41,767 INFO XGBoost-PySpark: _fit Running xgboost-3.2.0 on 3 workers with
	booster params: {'device': 'cpu', 'learning_rate': 0.05, 'max_depth': 4, 'objective': 'reg:squarederror', 'subsample': 0.7, 'nthread': 1}
	train_call_kwargs_params: {'verbose_eval': True, 'num_boost_round': 50}
	dmatrix_kwargs: {'nthread': 1, 'missing': nan}
2026-05-08 14:38:41,870 INFO XGBoost-PySpark: _fit Running xgboost-3.2.0 on 3 workers with
	booster params: {'device': 'cpu', 'learning_rate': 0.05, 'max_depth': 4, 'objective': 'reg:squarederror', 'subsample': 0.9, 'nthread': 1}
	train_call_kwargs_params: {'verbose_eval': True, 'num_boost_round': 50}
	dmatrix_kwargs: {'nthread': 1, 'missing': nan}
[14:38:47] [0]	training-rmse:28.30726 3][Stage 36:>                 (0 + 3) / 3]
[14:38:47] [1]	training-rmse:27.44911
[14:38:47] [0]	training-rmse:28.30411
[14:38:47] [2]	training-rmse:26.65051
[14:38:47] [1]	training-rmse:27.44810
[14:38:47] [3]	training-rmse:25.91136
[14:38:47] [2]	training-rms


Best XGBoost params:
  max_depth: 8
  learning_rate: 0.1
  n_estimators: 100
  subsample: 0.9



=== xgboost_tuned ===
  MAE:      6.9086
  RMSE:     13.5803
  WAPE:     43.34%
  MAPE(≥5): 53.44%
  R²:       0.7542


## 2. Random Forest Hyperparameter Tuning

Grid search trên:
- `numTrees`: số cây trong rừng
- `maxDepth`: độ sâu mỗi cây
- `minInstancesPerNode`: số mẫu tối thiểu mỗi node (tránh overfitting)

In [7]:
# ---- Random Forest Grid Search ----
rf = RandomForestRegressor(
    featuresCol="features",
    labelCol=TARGET_COL,
    predictionCol="prediction",
    seed=42,
)

rf_pipeline = Pipeline(stages=[assembler, rf])

rf_grid = (
    ParamGridBuilder()
    .addGrid(rf.numTrees,              [50, 100])
    .addGrid(rf.maxDepth,              [6, 10])
    .addGrid(rf.minInstancesPerNode,   [1, 10])
    .addGrid(rf.maxBins,               [32, 64])
    .build()
)

print(f"RF grid size: {len(rf_grid)} combinations × 3 folds = {len(rf_grid)*3} training runs")

cv_rf = CrossValidator(
    estimator=rf_pipeline,
    estimatorParamMaps=rf_grid,
    evaluator=RegressionEvaluator(labelCol=TARGET_COL, predictionCol="prediction", metricName="rmse"),
    numFolds=3,
    parallelism=2,
    seed=42,
)

print("Fitting RF CrossValidator... (may take 15–40 minutes)")
cv_rf_model = cv_rf.fit(train_hw)

best_rf_params = cv_rf_model.getEstimatorParamMaps()[int(np.argmin(cv_rf_model.avgMetrics))]
print("\nBest RF params:")
for k, v in best_rf_params.items():
    print(f"  {k.name}: {v}")

rf_pred = cv_rf_model.transform(test_hw)
rf_result = evaluate("random_forest_tuned", rf_pred)
results.append(rf_result)


RF grid size: 16 combinations × 3 folds = 48 training runs
Fitting RF CrossValidator... (may take 15–40 minutes)


26/05/08 14:42:22 WARN DAGScheduler: Broadcasting large task binary with size 1106.8 KiB
26/05/08 14:42:23 WARN DAGScheduler: Broadcasting large task binary with size 2.0 MiB
26/05/08 14:42:24 WARN DAGScheduler: Broadcasting large task binary with size 3.8 MiB
26/05/08 14:42:26 WARN DAGScheduler: Broadcasting large task binary with size 1083.6 KiB
26/05/08 14:42:26 WARN DAGScheduler: Broadcasting large task binary with size 1103.5 KiB
26/05/08 14:42:27 WARN DAGScheduler: Broadcasting large task binary with size 6.9 MiB
26/05/08 14:42:28 WARN DAGScheduler: Broadcasting large task binary with size 2047.3 KiB
26/05/08 14:42:29 WARN DAGScheduler: Broadcasting large task binary with size 1926.2 KiB
26/05/08 14:42:30 WARN DAGScheduler: Broadcasting large task binary with size 3.7 MiB
26/05/08 14:42:31 WARN DAGScheduler: Broadcasting large task binary with size 1031.7 KiB
26/05/08 14:42:32 WARN DAGScheduler: Broadcasting large task binary with size 6.7 MiB
26/05/08 14:42:34 WARN DAGScheduler:


Best RF params:
  numTrees: 100
  maxDepth: 10
  minInstancesPerNode: 1
  maxBins: 64



=== random_forest_tuned ===
  MAE:      7.2711
  RMSE:     13.9194
  WAPE:     45.61%
  MAPE(≥5): 56.96%
  R²:       0.7418


In [8]:
# ---- Summary: Baseline vs Tuned ----
summary = pd.DataFrame(results).sort_values("WAPE")
print("\n===== TUNING RESULTS (sorted by WAPE) =====")
print(summary.to_string(index=False))
display(summary)

best_model_name = summary.iloc[0]["model"]
print(f"\nBest tuned model: {best_model_name}")
print(f"  WAPE : {summary.iloc[0]['WAPE']:.2f}%")
print(f"  R²   : {summary.iloc[0]['R2']:.4f}")



===== TUNING RESULTS (sorted by WAPE) =====
              model      MAE      RMSE      WAPE  MAPE(>=5)       R2
      xgboost_tuned 6.908561 13.580349 43.339070  53.440908 0.754191
random_forest_tuned 7.271055 13.919393 45.613081  56.956654 0.741764


,model,MAE,RMSE,WAPE,MAPE(>=5),R2
0,xgboost_tuned,6.908561,13.580349,43.339070,53.440908,0.754191
1,random_forest_tuned,7.271055,13.919393,45.613081,56.956654,0.741764



Best tuned model: xgboost_tuned
  WAPE : 43.34%
  R²   : 0.7542


In [9]:
# ---- Save best model + metrics to HDFS ----
from datetime import datetime

run_id = datetime.utcnow().strftime("%Y%m%d_%H%M%S")
out_dir = f"{OUT_BASE_ROOT}/run_{run_id}"

# Save CV metrics as parquet
metrics_sdf = spark.createDataFrame(summary)
metrics_sdf.write.mode("overwrite").parquet(f"{out_dir}/metrics")

# Save best model
if best_model_name == "xgboost_tuned":
    cv_xgb_model.bestModel.write().overwrite().save(f"{out_dir}/best_model_xgb")
    print(f"Saved XGBoost best model to {out_dir}/best_model_xgb")
else:
    cv_rf_model.bestModel.write().overwrite().save(f"{out_dir}/best_model_rf")
    print(f"Saved RF best model to {out_dir}/best_model_rf")

meta = {
    "run_id": run_id,
    "feature_path": FEATURE_PATH,
    "best_model": best_model_name,
    "best_wape": float(summary.iloc[0]["WAPE"]),
    "best_r2": float(summary.iloc[0]["R2"]),
    "out_dir": out_dir,
}
print("\nRun Metadata:")
print(json.dumps(meta, indent=2))


26/05/08 14:48:32 WARN TaskSetManager: Stage 1743 contains a task of very large size (2478 KiB). The maximum recommended task size is 1000 KiB.


Saved XGBoost best model to /user/data/results/demand_prediction/tuning/run_20260508_144829/best_model_xgb

Run Metadata:
{
  "run_id": "20260508_144829",
  "feature_path": "/user/data/feature_engineering/demand_prediction_features",
  "best_model": "xgboost_tuned",
  "best_wape": 43.33906998158889,
  "best_r2": 0.7541912631497555,
  "out_dir": "/user/data/results/demand_prediction/tuning/run_20260508_144829"
}


26/05/08 15:02:16 ERROR TaskSchedulerImpl: Lost executor 0 on 172.18.0.2: Command exited with code 137
26/05/08 15:02:16 WARN BlockManagerMasterEndpoint: No more replicas available for rdd_32_7 !
26/05/08 15:02:16 WARN BlockManagerMasterEndpoint: No more replicas available for rdd_61_5 !
26/05/08 15:02:16 WARN BlockManagerMasterEndpoint: No more replicas available for rdd_19_10 !
26/05/08 15:02:16 WARN BlockManagerMasterEndpoint: No more replicas available for rdd_47_2 !
26/05/08 15:02:16 WARN BlockManagerMasterEndpoint: No more replicas available for rdd_19_11 !
26/05/08 15:02:16 WARN BlockManagerMasterEndpoint: No more replicas available for rdd_41_5 !
26/05/08 15:02:16 WARN BlockManagerMasterEndpoint: No more replicas available for rdd_47_8 !
26/05/08 15:02:16 WARN BlockManagerMasterEndpoint: No more replicas available for rdd_32_1 !
26/05/08 15:02:16 WARN BlockManagerMasterEndpoint: No more replicas available for rdd_47_11 !
26/05/08 15:02:16 WARN BlockManagerMasterEndpoint: No mor